# EDA of `aqi_hour` — Open Air Chicago, hourly

**STAT 473 · Bobo's notebook.** This notebook reproduces the *aqi_hour* EDA artifact step by step: every chart and table in the artifact comes from a cell below.

**How to run:** open it from the `code/` folder (same place as `analysis.ipynb` and `eda.ipynb`) with the `.venv` kernel, then *Run All*. The first cell with data reads the whole 880 MB CSV, so give it ~30 seconds. Nothing in `data/` is changed; every figure is also saved to `outputs/eda_bobo/`.

**Scope:** only `data/aqi_hour.csv` is analysed. `data/city_grid.geojson` is used once, as the street backdrop on the map. Weather is left out on purpose.

| Step | Question |
|---|---|
| 1 | Which columns actually have data? |
| 2 | What time zone are the timestamps in? |
| 3 | Where are the sensors, and which ones are "special"? |
| 4 | How complete is each sensor over time, and how long are the gaps? |
| 5 | Which time window should we analyse? |
| 6 | What do the NO₂ and PM2.5 distributions look like? |
| 7 | How often is the air unhealthy (AQI), and why? |
| 8 | What does the citywide picture look like over the year? |
| 9 | Do time anchors (hour of day, day of week) carry signal? |
| 10 | How much do lag features tell us? |
| 11 | How noisy are the sensors themselves? |
| 12 | What do we carry into feature building? |

Each step has three parts: **Why** (the reason we do it), the **code**, and **What we see** (the observations, written from the results of running this notebook on Sep 27 2026).

## 0. Setup
Imports, file paths, and the colours used throughout. NO₂ is always **blue** and PM2.5 is always **orange**, so a reader can tell the pollutant apart in any chart.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from IPython.display import display
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)   # e.g. mean of an empty slice

DATA = Path("data")
CSV = DATA / "aqi_hour.csv"
OUT = Path("outputs/eda_bobo")            # every figure is saved here too
OUT.mkdir(parents=True, exist_ok=True)

TZ = "America/Chicago"                     # the city's own clock (handles daylight saving time)
START = pd.Timestamp("2025-09-01", tz=TZ)  # dataset's official start; before this = installation period

# one colour per meaning, used in every chart
C = dict(no2="#2a78d6", pm="#eb6834", summer="#1baf7a", winter="#4a3aa7", grey="#5d6a70")
AQI_CATS = ["Good", "Moderate", "USG", "Unhealthy", "Very unhealthy", "Hazardous"]
AQI_COLORS = ["#00e400", "#ffff00", "#ff7e00", "#ff0000", "#8f3f97", "#7e0023"]   # official EPA colours

plt.rcParams.update({
    "figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.3, "axes.titleweight": "bold", "axes.titlesize": 11,
    "legend.frameon": False, "font.size": 10,
})

def save(fig, name):
    # save a figure as PNG into outputs/eda_bobo/
    fig.savefig(OUT / f"{name}.png", dpi=150, bbox_inches="tight")

def month_axis(ax):
    # nicer date axis: one tick per month, e.g. "Sep", "Jan '26"
    ax.xaxis.set_major_locator(mdates.MonthLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b"))

## Step 1 · Load the file and check what each column holds

**Why:** before any analysis we need to know which columns actually have data. A column that is 99% empty can't become a feature. We also confirm there is exactly one row per sensor per hour, because every later step (lags especially) depends on that.

We do this in two passes:
1. Read the whole file in chunks and count the non-empty cells in every column (this keeps memory low).
2. Load only the columns we need, with short names.

In [ ]:
# Pass 1: % filled for every column, read 500k rows at a time
n_rows, filled = 0, None
for chunk in pd.read_csv(CSV, chunksize=500_000, low_memory=False):
    n_rows += len(chunk)
    counts = chunk.notna().sum()
    filled = counts if filled is None else filled + counts

completeness = (filled / n_rows * 100).round(2)
print(f"{n_rows:,} rows x {len(completeness)} columns")
completeness.sort_values().to_frame("% filled")

The 54 columns fall into a few families. The chart below shows the average % filled for each family, which is easier to read than 54 separate bars.

In [ ]:
def column_group(col):
    # put each raw column into a readable family
    if col.startswith("bc"):                      return "Black carbon (10 cols)"
    if col.startswith(("temperatureAmbient", "relHumidAmbient", "wind", "atmPressure")):
                                                  return "Ambient weather"
    if col.startswith(("temperatureInternal", "relHumidInternal")):
                                                  return "Internal temp / humidity (inside the box)"
    if col.startswith("no2Conc1HourMeanUsEpaAqi"):return "NO2 AQI"
    if col.startswith("no2"):                     return "NO2 concentration"
    if "Nowcast" in col:                          return "PM2.5 NowCast + AQI"
    if col.startswith("pm2_5ConcMass1HourMean"):  return "PM2.5 concentration (1 h)"
    if col.startswith("pm2_5"):                   return "PM2.5 24 h rolling / number conc."
    return "IDs, time, location"

groups = completeness.groupby(completeness.index.map(column_group)).mean().sort_values()

fig, ax = plt.subplots(figsize=(8, 3.6))
ax.barh(groups.index, groups.values, color=C["grey"])
for y, v in enumerate(groups.values):
    ax.text(v + 1, y, f"{v:.1f}%", va="center", fontsize=9)
ax.set_xlim(0, 110); ax.set_xlabel("% of rows filled"); ax.grid(axis="y", visible=False)
ax.set_title("Step 1 · How complete is each column family?")
save(fig, "s1_completeness"); plt.show()

In [ ]:
# Pass 2: load only the columns this EDA needs and give them short names
COLS = {
    "datasourceid": "sid",           "sensor_name": "name",
    "startofperiod": "startofperiod",
    "latitude": "lat",               "longitude": "lon",
    "no2Conc1HourMean.value": "no2", "no2Conc1HourMean.raw": "no2_raw",
    "no2Conc1HourMeanUsEpaAqi.value": "no2_aqi",
    "pm2_5ConcMass1HourMean.value": "pm", "pm2_5ConcMass1HourMean.raw": "pm_raw",
    "pm2_5ConcMassNowcast.value": "pm_nowcast",
    "pm2_5ConcMassNowcastUsEpaAqi.value": "pm_aqi",
}
df = pd.read_csv(CSV, usecols=list(COLS), low_memory=False).rename(columns=COLS)

# sensor type (explained in step 3): collocation test units and O'Hare are not neighbourhood sensors
name = df["name"].astype(str)
df["kind"] = np.select([name.str.contains("Collocation"), name.str.startswith("O'Hare")],
                       ["collocation", "ohare"], default="community")
print(df.shape)
df.head()

**What we see**
- NO₂ (`.value`) is filled in 99.5% of rows and PM2.5 in 98.8%, so the two pollutants we care about are nearly complete.
- Each pollutant comes as `.raw` and `.value`. `.value` is the **calibrated** number, and we use it (step 6 shows why).
- All 10 black-carbon columns are empty.
- The **ambient weather** columns (outdoor temperature, humidity, wind, pressure) are only 0.8% filled. Internal temperature/humidity are 99.5% filled, but they're measured *inside the sensor box*, so they aren't weather. Any weather features will need a separate source.

## Step 2 · Find out what time zone the timestamps use

**Why:** our teammates want time features ("time of day", "day of week"). Those only mean something in Chicago clock time. If the file is in UTC and we treat it as local time, every hour feature is off by 5–6 hours, and the error changes when daylight saving time starts or ends.

We test three things:
1. **Daylight saving days.** In Chicago clock time, Nov 2 2025 has 25 hours and Mar 8 2026 has 23. If the file were in local time, those days would show 25 and 23 distinct hours.
2. **Rush-hour peak.** NO₂ is traffic-driven and should peak during the morning commute (6–8 AM).
3. **July 4th.** Fireworks should show up as a PM2.5 spike around 9 PM.

In [ ]:
# parse as UTC, then convert to Chicago time (tz_convert handles daylight saving time)
df["utc"] = (pd.to_datetime(df["startofperiod"], format="%m/%d/%Y %I:%M:%S %p")
               .dt.tz_localize("UTC"))
df["local"] = df["utc"].dt.tz_convert(TZ)
df = df.drop(columns="startofperiod")

print("unparsed timestamps:", df["utc"].isna().sum())
print("timestamps not on the hour:", (df["utc"].dt.minute != 0).sum())
print("duplicate sensor-hours:", df.duplicated(["sid", "utc"]).sum())

# Test 1: daylight saving days, using the hours exactly as stored
stored = df["utc"].dt.tz_localize(None)
for d in ["2025-11-02", "2026-03-08"]:
    day = stored[stored.dt.normalize() == pd.Timestamp(d)]
    per_hour = day.dt.hour.value_counts()
    print(f"{d}: {day.dt.hour.nunique()} distinct hours, rows per hour min/max = {per_hour.min()}/{per_hour.max()}")

print("file range (Chicago time):", df["local"].min(), "->", df["local"].max())

In [ ]:
# Test 2: weekday NO2 profile by stored hour vs Chicago hour, summer vs winter
com = df[(df["kind"] == "community") & (df["local"] >= START)]
wkday = com[com["local"].dt.dayofweek < 5]
month = wkday["local"].dt.month
seasons = {"Summer (Jun–Aug)": (month.isin([6, 7, 8]), C["summer"]),
           "Winter (Dec–Feb)": (month.isin([12, 1, 2]), C["winter"])}

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8), sharey=True)
for label, (mask, color) in seasons.items():
    sub = wkday[mask]
    axes[0].plot(sub.groupby(sub["utc"].dt.hour)["no2"].mean(), color=color, lw=2, label=label)
    axes[1].plot(sub.groupby(sub["local"].dt.hour)["no2"].mean(), color=color, lw=2, label=label)
axes[0].set(title="Hours as stored in the file", xlabel="stored hour", ylabel="mean NO₂ (ppb)")
axes[1].set(title="After converting to Chicago time", xlabel="hour (Chicago)")
for ax in axes:
    ax.set_xticks(range(0, 24, 3)); ax.legend()
fig.suptitle("Step 2 · Weekday NO₂ by hour, community sensors", y=1.02, fontweight="bold")
save(fig, "s2_timezone"); plt.show()

# Test 3: July 4 2026, citywide median PM2.5 by Chicago hour
j4 = com[com["local"].dt.normalize() == pd.Timestamp("2026-07-04", tz=TZ)]
print(j4.groupby(j4["local"].dt.hour)["pm"].median().round(1).to_dict())

**What we see**
- Both daylight saving days have exactly **24** hours with the same number of rows every hour, so the clock never jumps. The file is not in local time.
- In the stored hours, the weekday NO₂ peak is at hour **11 in summer** and **13 in winter**. That only makes sense as UTC: 11 UTC = 6 AM CDT and 13 UTC = 7 AM CST.
- After converting, both seasons peak at 6–7 AM and bottom out around noon to 1 PM. On July 4, PM2.5 jumps from about 12 to 89 µg/m³ at **9 PM**, which is fireworks time.
- **Conclusion:** `startofperiod` is **UTC**. We use the `local` column for every time feature. Every timestamp is on the hour and there are no duplicate sensor-hours.

## Step 3 · Where the sensors are

**Why:** we need to know what each sensor measures and where it sits before we pool them. Some units aren't neighbourhood sensors: they're mounted in groups of three to test the sensors themselves (**collocation** sites), or placed at **O'Hare**. Mixing them in would put extra weight on a few spots. The map also shows whether coverage is even across the city.

In [ ]:
post = df[df["local"] >= START]           # means are computed on the analysis period only

sensors = (df.groupby("sid")
             .agg(name=("name", "first"), kind=("kind", "first"),
                  lat=("lat", "first"), lon=("lon", "first"),
                  n_names=("name", "nunique"), lat_sd=("lat", "std"), lon_sd=("lon", "std"),
                  first=("local", "min"), last=("local", "max"), n_hours=("utc", "size"))
             .join(post.groupby("sid")[["no2", "pm"]].mean().add_suffix("_mean")))

span_hours = (sensors["last"] - sensors["first"]) / pd.Timedelta("1h") + 1
sensors["pct_of_span"] = (sensors["n_hours"] / span_hours * 100).round(1)

print(sensors["kind"].value_counts().to_string())
print("max names per sensor id:", sensors["n_names"].max())
print("sensors whose coordinates change:", ((sensors["lat_sd"] > 1e-6) | (sensors["lon_sd"] > 1e-6)).sum())

The map uses the Chicago street centre lines as a backdrop (grey lines; expressways darker). Community sensors are coloured dots. Collocation and O'Hare units are hollow squares. Colour limits are set at the 2nd and 98th percentiles so one extreme sensor doesn't wash out the rest.

If `geopandas` isn't installed, the map still draws, just without streets.

In [ ]:
try:
    import geopandas as gpd
    streets = gpd.read_file(DATA / "city_grid.geojson")           # backdrop only (~20 s)
    streets["class"] = streets["class"].astype(str)
except Exception as e:
    streets = None
    print("no street backdrop:", e)

comm = sensors[sensors["kind"] == "community"]
other = sensors[sensors["kind"] != "community"]

fig, axes = plt.subplots(1, 2, figsize=(13, 8.5))
for ax, col, cmap, label in [(axes[0], "no2_mean", "Blues", "mean NO₂ (ppb)"),
                             (axes[1], "pm_mean", "Oranges", "mean PM2.5 (µg/m³)")]:
    if streets is not None:
        streets.plot(ax=ax, color="#d5dadb", lw=0.2)                              # all streets = city shape
        streets[streets["class"].isin(["2", "3"])].plot(ax=ax, color="#b3bcbe", lw=0.5)  # arterials
        streets[streets["class"] == "1"].plot(ax=ax, color="#7d888c", lw=1.2)            # expressways
    lo, hi = comm[col].quantile([0.02, 0.98])
    sc = ax.scatter(comm["lon"], comm["lat"], c=comm[col], cmap=cmap, vmin=lo, vmax=hi,
                    s=28, edgecolor="white", lw=0.6, zorder=3)
    ax.scatter(other["lon"], other["lat"], marker="s", s=40, facecolor="none",
               edgecolor="black", lw=1.2, zorder=4, label="collocation / O'Hare")
    ax.set_aspect(1 / np.cos(np.radians(41.84)))       # keeps Chicago's real shape
    ax.set(xlim=(-87.95, -87.51), ylim=(41.635, 42.03), xticks=[], yticks=[])
    ax.grid(False)
    fig.colorbar(sc, ax=ax, shrink=0.6, label=label)
    ax.legend(loc="lower left")
axes[0].set_title("Mean NO₂ per sensor, Sep 2025 – Sep 2026")
axes[1].set_title("Mean PM2.5 per sensor, Sep 2025 – Sep 2026")
fig.suptitle("Step 3 · Sensor locations", fontweight="bold")
save(fig, "s3_map"); plt.show()

In [ ]:
# highest and lowest mean NO2 among community sensors
ranked = comm.sort_values("no2_mean", ascending=False)[["name", "no2_mean", "pm_mean", "pct_of_span"]].round(1)
display(pd.concat([ranked.head(5), ranked.tail(5)]))
print(comm[["no2_mean", "pm_mean"]].describe().round(2))

**What we see**
- 286 sensors: **274 community**, 9 collocation (3 each at Mayfair, Schiller Park and Washington HS) and 3 at O'Hare. From step 5 on we use only the 274 community sensors.
- No sensor ever changes coordinates, and each ID has exactly one name.
- Every part of the city has sensors, with more on the South and West Sides.
- Mean NO₂ ranges from 10.4 to 25.8 ppb. The highest cluster on the Southwest Side (New City, Brighton Park, the Lawndales, Archer Heights) and the Near West Side.
- PM2.5 is much more even (5.2–18.4 µg/m³, SD 1.1). **Avalon Park 1** stands out at 18.4, about 7 µg/m³ above the typical sensor. That could be a local source or a faulty sensor, so it's worth a closer look.

## Step 4 · Coverage and gaps over time

**Why:** a lag feature such as "NO₂ one hour ago" needs the previous hour to exist. If sensors drop out often, many rows lose their lags, and a plain `shift(1)` on the sorted rows silently grabs the *wrong* hour. So we measure when sensors came online, how complete each one is, and how long the gaps are.

In [ ]:
# how many sensors report each (Chicago) day
day = df["local"].dt.normalize()
sensors_per_day = df.groupby(day)["sid"].nunique()

fig, ax = plt.subplots(figsize=(12, 3.2))
ax.plot(sensors_per_day.index, sensors_per_day.values, color=C["grey"], lw=1.5)
ax.axvspan(sensors_per_day.index.min(), START, color="0.9", zorder=0, label="installation period")
ax.set(ylim=(0, 300), ylabel="sensors reporting", title="Step 4 · Sensors reporting each day")
month_axis(ax); ax.legend(loc="lower right")
save(fig, "s4_sensors_per_day"); plt.show()

# find big one-day drops (network outages)
drop = sensors_per_day.diff()
print("days where 50+ sensors disappeared at once:")
print(drop[drop <= -50])
print(sensors_per_day["2026-07-26":"2026-08-15"].to_string())

In [ ]:
# weekly availability heatmap: one row per sensor, colour = share of the week's 168 hours with NO2
week = df["local"].dt.tz_localize(None).dt.to_period("W").dt.start_time
avail = (df.assign(week=week)
           .groupby(["sid", "week"])["no2"].count().div(168)
           .unstack(fill_value=0).clip(upper=1))
avail = avail.reindex(sensors.sort_values("first").index)     # rows sorted by first day online

x0 = mdates.date2num(avail.columns.min())
x1 = mdates.date2num(avail.columns.max() + pd.Timedelta("7D"))
fig, ax = plt.subplots(figsize=(12, 5))
im = ax.imshow(avail.values, aspect="auto", cmap="Blues", vmin=0, vmax=1,
               interpolation="nearest", extent=[x0, x1, len(avail), 0])
ax.xaxis_date(); month_axis(ax); ax.grid(False)
ax.set(ylabel="sensor (sorted by first day online)", title="Step 4 · Weekly NO₂ availability per sensor")
fig.colorbar(im, ax=ax, label="share of hours present")
save(fig, "s4_availability"); plt.show()

Now the **hourly grid**. For the analysis set (community sensors from Sep 1 2025; step 5 explains why) we build a table with one row per sensor and one column for *every* hour, so a missing hour becomes an explicit empty cell. We reuse this grid for gaps here and for lags and autocorrelation in step 10.

In [ ]:
A = df[(df["kind"] == "community") & (df["local"] >= START)].copy()     # the analysis set

hours = pd.date_range(A["utc"].min(), A["utc"].max(), freq="h")         # every hour, no holes
grid = {c: A.pivot(index="sid", columns="utc", values=c).reindex(columns=hours)
        for c in ["no2", "pm", "pm_aqi"]}
print("grid shape (sensors x hours):", grid["no2"].shape)

# --- gaps inside each sensor's active period (first to last reading)
obs = grid["no2"].notna().to_numpy()
gaps, coverage = [], []
for row in obs:
    idx = np.flatnonzero(row)
    active = row[idx[0]: idx[-1] + 1]
    coverage.append(active.mean())
    missing = np.r_[0, (~active).astype(int), 0]
    edges = np.diff(missing)
    gaps += list(np.flatnonzero(edges == -1) - np.flatnonzero(edges == 1))   # length of each run of missing hours
gaps = pd.Series(gaps)

bins   = [1, 2, 4, 7, 25, 169, np.inf]
labels = ["1 h", "2–3 h", "4–6 h", "7–24 h", "1–7 days", "> 7 days"]
gap_table = (pd.DataFrame({"length": gaps, "bin": pd.cut(gaps, bins, right=False, labels=labels)})
               .groupby("bin", observed=False)["length"].agg(gaps="size", missing_hours="sum"))
gap_table["% of missing hours"] = (gap_table["missing_hours"] / gap_table["missing_hours"].sum() * 100).round(1)
print(f"median sensor coverage: {np.median(coverage):.2%}")
display(gap_table)

fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
ax[0].bar(labels, gap_table["gaps"], color=C["grey"]);          ax[0].set_title("Number of gaps")
ax[1].bar(labels, gap_table["missing_hours"], color=C["grey"]); ax[1].set_title("Missing hours they cause")
for a in ax: a.tick_params(axis="x", rotation=30); a.grid(axis="x", visible=False)
fig.suptitle("Step 4 · Gap lengths (NO₂, community sensors)", fontweight="bold", y=1.03)
save(fig, "s4_gaps"); plt.show()

In [ ]:
# --- share of rows whose lag k exists (both hour t and hour t-k observed)
lag_avail = {f"{k} h": (obs[:, k:] & obs[:, :-k]).sum() / obs[:, k:].sum() for k in [1, 2, 3, 6, 24, 168]}
lag_avail["1, 2 and 3 h together"] = (obs[:, 3:] & obs[:, 2:-1] & obs[:, 1:-2] & obs[:, :-3]).sum() / obs[:, 3:].sum()
display(pd.Series(lag_avail).mul(100).round(2).to_frame("% of rows with lag present"))

# --- blank values inside rows that exist, by month
A["month"] = A["local"].dt.strftime("%Y-%m")
display(A.groupby("month")[["no2", "pm"]].apply(lambda g: g.isna().mean() * 100).round(2)
          .rename(columns={"no2": "% NO2 blank", "pm": "% PM2.5 blank"}))

# sensors that are much less complete than the rest
display(sensors.loc[(sensors["kind"] == "community") & (sensors["pct_of_span"] < 90),
                    ["name", "first", "pct_of_span"]])

**What we see**
- Sensors were installed in batches between Jul 26 and Aug 31 2025 (17 → 68 → 134 → 187 → 238 → 271). From September on, 271–286 sensors report every day, with one exception:
- **A network-wide outage on Jul 28 2026.** About 118 sensors stopped reporting at once (283 → 165). They came back gradually and were all back by Aug 13. So many sensors dropping out together points to a network or data-pipeline problem, not individual sensor failures. Those weeks will have many missing lags, and they fall inside any summer 2026 test period.
- Within its active period, the median community sensor has **99.75%** of its hours. Six sensors are much less complete: South Lawndale 4 (26%), South Deering 9 (39%), Loop 1 (52%), Hyde Park 2 and Uptown 2 (78%), and West Lawn 4 (87%).
- Most gaps are short (2–3 h), but most of the **missing hours** come from 117 long outages of more than a week.
- **99.8%** of rows have the 1-hour lag and 98.8% have the 1-week lag. Lags lose almost no data, as long as they're built on the full hourly grid.

## Step 5 · Choose the study window: drop the installation period

**Why:** Jul 26 – Aug 31 2025 was the installation period. The portal lists Sept 1 2025 as the official start, and our earlier daily analysis found that sensor offsets in those weeks didn't match later months. Few sensors were online, and they were still settling. Pooling those weeks would mix two different measurement situations.

In [ ]:
c_all = df[df["kind"] == "community"].copy()
c_all["period"] = np.where(c_all["local"] < START, "1 install (Jul 26 – Aug 31 2025)", "2 analysis (Sep 2025 – Sep 2026)")

def p99(s):
    return s.quantile(0.99)

summary = c_all.groupby("period")[["no2", "pm"]].agg(["count", "mean", "median", p99, "std"]).round(2)
display(summary)
print(f"analysis set: {len(A):,} sensor-hours from {A['sid'].nunique()} community sensors")

**What we see**
- The installation weeks have lower NO₂ (mean 12.4 vs 15.3 ppb) and a different PM2.5 mix. Part of that is simply summer, but those weeks also come from a changing set of sensors, so we can't separate the two.
- **Decision:** from here on, the analysis set `A` is the **274 community sensors from Sep 1 2025 to Sep 26 2026**: 2,501,376 sensor-hours.

## Step 6 · Distributions: shape, tails and bad values

**Why:** the shape of the data decides how we model it. A heavy right tail means a few extreme hours will dominate a squared-error model, and a log scale may be better. Low-cost sensors can also report impossible values (negatives) or get stuck on one number, and those need cleaning before we build lags.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 6.5))
# linear scale; PM2.5 is clipped at 80 so the shape is visible (the last bar holds every hour >= 79)
axes[0, 0].hist(A["no2"].dropna(), bins=np.arange(0, 81), color=C["no2"])
axes[0, 0].set(title="NO₂, hourly (ppb)", ylabel="sensor-hours")
axes[0, 1].hist(A["pm"].dropna().clip(upper=79.5), bins=np.arange(0, 81), color=C["pm"])
axes[0, 1].set(title="PM2.5, hourly (µg/m³) — last bar = all hours ≥ 79")
# log scale
axes[1, 0].hist(np.log10(A["no2"][A["no2"] > 0]), bins=70, color=C["no2"])
axes[1, 0].set(title="NO₂ on a log scale", xlabel="log10(ppb)", ylabel="sensor-hours")
axes[1, 1].hist(np.log10(A["pm"][A["pm"] > 0]), bins=70, color=C["pm"])
axes[1, 1].set(title="PM2.5 on a log scale", xlabel="log10(µg/m³)")
for ax in axes.flat: ax.grid(axis="x", visible=False)
fig.suptitle("Step 6 · Distributions (analysis set)", fontweight="bold")
fig.tight_layout(); save(fig, "s6_distributions"); plt.show()

In [ ]:
# quantiles, negatives and skew for calibrated (.value) vs raw columns
qs = [0, .01, .25, .5, .75, .99, .999, 1]
qtab = A[["no2", "no2_raw", "pm", "pm_raw"]].quantile(qs).T
qtab.columns = ["min", "1%", "25%", "median", "75%", "99%", "99.9%", "max"]
qtab["negatives"] = (A[["no2", "no2_raw", "pm", "pm_raw"]] < 0).sum()
qtab["zeros"] = (A[["no2", "no2_raw", "pm", "pm_raw"]] == 0).sum()
qtab["skew"] = A[["no2", "no2_raw", "pm", "pm_raw"]].skew()
display(qtab.round(2))

# how closely do raw and calibrated agree?
for p in ["no2", "pm"]:
    both = A[[p, p + "_raw"]].dropna()
    print(f"{p}: r(raw, value) = {both.corr().iloc[0, 1]:.2f},  median(value - raw) = {(both[p] - both[p + '_raw']).median():.2f}")

# stuck-sensor check: the same value repeated 6+ hours in a row (on the full hourly grid)
for p in ["no2", "pm"]:
    g = grid[p].to_numpy()
    same = (g[:, 1:] == g[:, :-1])                 # NaN never equals NaN, so gaps break runs
    longest = 0
    for row in same:
        run = 0
        for s in row:
            run = run + 1 if s else 0
            longest = max(longest, run)
    print(f"{p}: longest run of identical consecutive values = {longest + 1 if longest else 1} hours")

**What we see**
- **NO₂** (calibrated) is right-skewed but not extreme: median 13.9 ppb, 99th percentile 34.4, max 61.7. There are no zeros or negatives.
- **PM2.5** has a very long tail: median 8.5 and 99th percentile 34.9 µg/m³, but the 99.9th percentile is 264 and the max is 487 (skew 14.5). On a log scale it's close to symmetric, so **log PM2.5** is a better modelling scale.
- **Raw vs calibrated:** raw NO₂ is negative in 6.1% of hours (152,240) and only moderately correlated with the calibrated value (r = 0.70). Raw PM2.5 has 7,144 exact zeros and reads about 3.5 µg/m³ higher (r = 0.94). We use the calibrated `.value` columns.
- No stuck sensors: no run of 6 or more identical hourly values.

## Step 7 · How often is the air unhealthy? (AQI categories)

**Why:** one project goal is to flag unhealthy AQI hours ahead of time. Before building that, we need to know how rare those hours are, which pollutant causes them, and how the AQI column is calculated. All three affect what the target should be.

The EPA AQI categories are: 0–50 Good, 51–100 Moderate, 101–150 Unhealthy for Sensitive Groups (USG), 151–200 Unhealthy, 201–300 Very unhealthy, 301+ Hazardous.

In [ ]:
def aqi_category(a):
    return pd.cut(a, [-np.inf, 50, 100, 150, 200, 300, np.inf], labels=AQI_CATS)

shares = pd.DataFrame({
    "NO₂ AQI":   aqi_category(A["no2_aqi"].dropna()).value_counts(normalize=True),
    "PM2.5 AQI": aqi_category(A["pm_aqi"].dropna()).value_counts(normalize=True),
}).reindex(AQI_CATS).fillna(0) * 100
display(shares.round(3))

# 100% stacked horizontal bars in the official EPA colours
fig, ax = plt.subplots(figsize=(11, 2.4))
left = np.zeros(2)
for cat, color in zip(AQI_CATS, AQI_COLORS):
    ax.barh(shares.columns, shares.loc[cat], left=left, color=color, edgecolor="white", lw=0.8, label=cat)
    left += shares.loc[cat].values
ax.set(xlim=(0, 100), xlabel="% of sensor-hours", title="Step 7 · Share of hours in each AQI category")
ax.legend(ncol=6, bbox_to_anchor=(0.5, -0.45), loc="upper center", fontsize=8.5); ax.grid(False)
save(fig, "s7_aqi_categories"); plt.show()

both = A[["no2_aqi", "pm_aqi"]].dropna()
print(f"PM2.5 AQI higher than NO2 AQI in {(both['pm_aqi'] > both['no2_aqi']).mean():.1%} of hours")
print(f"sensor-hours with PM2.5 AQI > 100: {(A['pm_aqi'] > 100).sum():,} ({(A['pm_aqi'] > 100).mean():.2%})")

In [ ]:
# which days had the most unhealthy hours?
bad = A[A["pm_aqi"] > 100]
episodes = (bad.groupby(bad["local"].dt.date)
               .agg(sensor_hours_over_100=("sid", "size"), sensors=("sid", "nunique"))
               .sort_values("sensor_hours_over_100", ascending=False))
print("days with at least one hour > 100:", len(episodes))
display(episodes.head(10))

# check the AQI breakpoints: what AQI does a given NowCast value get?
for lo, hi in [(8.5, 9.0), (9.1, 9.6), (35.0, 35.4), (35.5, 36.0)]:
    sub = A.loc[A["pm_nowcast"].between(lo, hi), "pm_aqi"]
    print(f"NowCast {lo}-{hi} µg/m³ -> AQI {sub.min():.0f}-{sub.max():.0f}")

In [ ]:
# zoom into the biggest episode: hourly citywide PM2.5, Jul 13-22 2026
wk = A[(A["local"] >= pd.Timestamp("2026-07-13", tz=TZ)) & (A["local"] < pd.Timestamp("2026-07-23", tz=TZ))]
g = wk.groupby("local")["pm"]
fig, ax = plt.subplots(figsize=(12, 3.5))
ax.fill_between(g.median().index, g.quantile(.1), g.quantile(.9), color=C["pm"], alpha=.2, label="10th–90th pct of sensors")
ax.plot(g.median(), color=C["pm"], lw=1.8, label="median")
ax.axhline(35.5, color="0.4", ls=":", lw=1); ax.text(wk["local"].min(), 40, "AQI 100 (NowCast 35.5)", fontsize=8, color="0.3")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d", tz=TZ))
ax.set(ylabel="PM2.5 (µg/m³)", title="Step 7 · Jul 13–22 2026, hourly PM2.5"); ax.legend()
save(fig, "s7_july_episode"); plt.show()

**What we see**
- **NO₂ is never the problem.** 99.996% of NO₂ AQI hours are "Good": 107 "Moderate" hours and none above that. PM2.5's AQI is higher in 99.6% of hours, so the overall AQI here is really the PM2.5 AQI.
- **PM2.5:** 53% Good, 46% Moderate, and **0.87%** (21,823 sensor-hours) above 100, spread over 173 days. A few episodes account for most of those hours:
  - a citywide episode on **Jul 16–20 2026**, when 237–270 sensors were above 100 at once and the median reached several hundred µg/m³ (it looks like wildfire smoke, but this file can't confirm the source)
  - the **July 4–5** fireworks
  - **Feb 15 2026** and **Dec 2 2025**, both winter build-ups
- The PM2.5 AQI uses the **2024 EPA breakpoints** (NowCast 9.0 → AQI ≤ 50, 35.5 → 101). It's built from the **NowCast**, a weighted average of the past 12 hours, not from the 1-hour value. So the AQI is already partly a lag feature, which matters in step 10.

## Step 8 · The citywide picture over the year

**Why:** a time plot shows seasonality, trends and episodes that single numbers hide. It also tells us whether a train/test split by date will be fair: if one season looks very different, a model trained without it will struggle.

Method: for each sensor and day, take the daily mean. Then, for each day, take the median across sensors (line) and the 10th–90th percentile across sensors (band). All community sensors are shown, and the installation period is shaded.

In [ ]:
c_all["day"] = c_all["local"].dt.tz_localize(None).dt.normalize()
sensor_day = c_all.groupby(["day", "sid"])[["no2", "pm"]].mean()
by_day = sensor_day.groupby("day")

fig, axes = plt.subplots(2, 1, figsize=(12, 6.5), sharex=True)
for ax, p, unit, top in [(axes[0], "no2", "ppb", None), (axes[1], "pm", "µg/m³", 60)]:
    med, lo, hi = by_day[p].median(), by_day[p].quantile(.1), by_day[p].quantile(.9)
    ax.fill_between(med.index, lo, hi, color=C[p], alpha=.2, label="10th–90th pct of sensors")
    ax.plot(med, color=C[p], lw=1.4, label="median")
    ax.axvspan(med.index.min(), START.tz_localize(None), color="0.9", zorder=0, label="installation period")
    ax.set(ylabel=f"{'NO₂' if p == 'no2' else 'PM2.5'} ({unit})")
    if top: ax.set_ylim(0, top)
    ax.legend(loc="upper left", ncol=3)
axes[0].set_title("Step 8 · Daily NO₂")
axes[1].set_title("Daily PM2.5 (y-axis cut at 60; the July episode goes beyond it)")
month_axis(axes[1])
fig.tight_layout(); save(fig, "s8_daily"); plt.show()

# monthly means (analysis set)
display(A.groupby("month")[["no2", "pm"]].mean().round(2).T)

**What we see**
- NO₂ is higher in winter: monthly means of 17.5–18.8 ppb in Dec–Feb versus 13.7–14.9 in summer and early fall. Colder air holds pollution near the ground.
- PM2.5 has no smooth seasonal curve. It's mostly a quiet baseline of 5–10 µg/m³, broken by multi-day episodes in winter (Dec, Feb) and summer (July).
- On most days all sensors rise and fall together, and the band is narrow compared with the day-to-day swings. That matches step 9, where most of the variance turns out to be citywide.

## Step 9 · Time anchors: hour of day and day of week

**Why:** our teammates proposed time-of-day and day-of-week features. Here we check whether they carry real signal and how much. Traffic follows a clock and a weekly schedule, so NO₂ should respond. PM2.5 is more regional and may not.

We remove **citywide episode hours** (hours when the median PM2.5 across sensors is above 35 µg/m³) so a few extreme hours don't distort the typical profile.

In [ ]:
# time-anchor columns (all in Chicago time)
A["hour"] = A["local"].dt.hour
A["dow"] = A["local"].dt.dayofweek            # 0 = Monday ... 6 = Sunday
A["weekend"] = A["dow"] >= 5
A["season"] = A["local"].dt.month.map({12: "winter", 1: "winter", 2: "winter", 3: "spring", 4: "spring", 5: "spring",
                                       6: "summer", 7: "summer", 8: "summer", 9: "fall", 10: "fall", 11: "fall"})

# flag citywide episode hours and drop them for the profiles
city_pm = A.groupby("utc")["pm"].median()
episode_hours = city_pm.index[city_pm > 35]
A["episode"] = A["utc"].isin(episode_hours)
B = A[~A["episode"]]
print("citywide episode hours removed:", len(episode_hours))

In [ ]:
DOW = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
fig, axes = plt.subplots(2, 2, figsize=(12, 7))

# hour-of-day profiles, weekday (solid) vs weekend (dashed)
for ax, p, unit in [(axes[0, 0], "no2", "ppb"), (axes[0, 1], "pm", "µg/m³")]:
    prof = B.pivot_table(index="hour", columns="weekend", values=p, aggfunc="mean")
    ax.plot(prof.index, prof[False], color=C[p], lw=2, label="weekday")
    ax.plot(prof.index, prof[True], color=C[p], lw=2, ls="--", label="weekend")
    ax.set(xticks=range(0, 24, 3), xlabel="hour (Chicago)", ylabel=unit,
           title=f"{'NO₂' if p == 'no2' else 'PM2.5'} by hour of day"); ax.legend()

# day-of-week means (bars start at 0 so heights are honest)
for ax, p, unit in [(axes[1, 0], "no2", "ppb"), (axes[1, 1], "pm", "µg/m³")]:
    m = B.groupby("dow")[p].mean()
    ax.bar(DOW, m.values, color=C[p])
    for x, v in enumerate(m.values): ax.text(x, v + 0.2, f"{v:.1f}", ha="center", fontsize=8.5)
    ax.set(ylabel=unit, title=f"{'NO₂' if p == 'no2' else 'PM2.5'} by day of week"); ax.grid(axis="x", visible=False)

fig.suptitle("Step 9 · Time anchors (episode hours removed)", fontweight="bold")
fig.tight_layout(); save(fig, "s9_profiles"); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 3.8), gridspec_kw={"width_ratios": [1.3, 1]})

# day x hour heatmap for NO2
hm = B.pivot_table(index="dow", columns="hour", values="no2", aggfunc="mean")
im = axes[0].imshow(hm.values, aspect="auto", cmap="Blues")
axes[0].set(yticks=range(7), yticklabels=DOW, xticks=range(0, 24, 3), xlabel="hour (Chicago)",
            title="NO₂: day × hour (mean ppb)"); axes[0].grid(False)
fig.colorbar(im, ax=axes[0], label="ppb")

# NO2 profile, summer vs winter
for s, col in [("summer", C["summer"]), ("winter", C["winter"])]:
    axes[1].plot(B[B["season"] == s].groupby("hour")["no2"].mean(), color=col, lw=2, label=s.title())
axes[1].set(xticks=range(0, 24, 3), xlabel="hour (Chicago)", ylabel="ppb", title="NO₂ by hour, summer vs winter")
axes[1].legend()
fig.tight_layout(); save(fig, "s9_heatmap_seasons"); plt.show()

### How much of the variance does each factor explain?
For a factor such as "hour of day", replace every value by the mean of its group (e.g. the mean for 7 AM). The share of the total variance that those group means reproduce is the R² of that factor alone. We compare time anchors with two stronger references:
- **Citywide, each hour:** the average of all sensors at that exact hour (this captures weather and regional air).
- **Sensor:** each sensor's own long-run level (this captures location).

In [ ]:
def r2_of(data, keys, col):
    # share of variance explained by the group means of `keys`
    x = data.dropna(subset=[col])
    fitted = x.groupby(keys)[col].transform("mean")
    return fitted.var(ddof=0) / x[col].var(ddof=0)

def r2_time_plus_sensor(data, col):
    # citywide hour mean + sensor offset (additive): 1 - var(residual) / var(total)
    x = data.dropna(subset=[col])
    r = x[col] - x.groupby("utc")[col].transform("mean")
    r = r - r.groupby(x["sid"]).transform("mean")
    return 1 - r.var(ddof=0) / x[col].var(ddof=0)

factors = {"Hour of day": "hour", "Day of week": "dow", "Hour × day": ["hour", "dow"],
           "Month": "month", "Hour × day × month": ["hour", "dow", "month"],
           "Sensor (location)": "sid", "Citywide, each hour": "utc"}
vd = pd.DataFrame({p: {lab: r2_of(B, k, p) for lab, k in factors.items()} for p in ["no2", "pm"]})
vd.loc["Citywide hour + sensor"] = [r2_time_plus_sensor(B, p) for p in ["no2", "pm"]]
display((vd * 100).round(1).rename(columns={"no2": "NO2 %", "pm": "PM2.5 %"}))

fig, ax = plt.subplots(figsize=(10, 4.2))
y = np.arange(len(vd))
ax.barh(y - 0.2, vd["no2"] * 100, height=0.38, color=C["no2"], label="NO₂")
ax.barh(y + 0.2, vd["pm"] * 100, height=0.38, color=C["pm"], label="PM2.5")
for i, (a, b) in enumerate(zip(vd["no2"], vd["pm"])):
    ax.text(a * 100 + 0.8, i - 0.2, f"{a:.1%}", va="center", fontsize=8)
    ax.text(b * 100 + 0.8, i + 0.2, f"{b:.1%}", va="center", fontsize=8)
ax.set(yticks=y, yticklabels=vd.index, xlim=(0, 100), xlabel="% of hourly variance explained")
ax.invert_yaxis(); ax.legend(loc="lower right"); ax.grid(axis="y", visible=False)
ax.set_title("Step 9 · How much each factor explains on its own")
save(fig, "s9_variance"); plt.show()

**What we see**
- **NO₂ has a clear daily cycle.** On weekdays it peaks at 7 AM (18.8 ppb) and bottoms out around 1 PM (13.3), when the afternoon air mixes best, then rises again in the evening. Weekends are about 1–2 ppb lower all day, with a much smaller morning peak.
- **NO₂ by day:** Tuesday to Thursday are highest (15.3–16.3 ppb), Saturday is 14.4 and Sunday 13.8. That's a traffic signature.
- **PM2.5 hardly moves by hour or by day.** There's a small 7 AM bump on weekdays, the whole daily range is about 1.8 µg/m³, and weekends are about 0.5 lower. Day-of-week means stay between 9.7 and 10.7 with no clear pattern.
- In winter, NO₂ is higher at every hour and the evening build-up starts earlier (earlier sunset).
- **Where the variance lives:** hour + weekday explain only **8%** of NO₂ and **1.4%** of PM2.5, and adding the month raises that to 24% / 29%. The citywide mean at that exact hour explains **58%** of NO₂ and **84%** of PM2.5, and adding each sensor's level takes it to 70% / 86%.
- **Takeaway:** time anchors are useful for NO₂ but secondary. Most of the variation is hour-to-hour citywide conditions, and that's what lag features pick up.

## Step 10 · Lag features: how much does the past tell us?

**Why:** a lag feature is a past value of the same sensor, such as NO₂ one hour ago. The **autocorrelation function (ACF)** shows how strongly the value at hour *t* correlates with hour *t − k*, for each lag *k*. That tells us which lags are worth keeping. We then compare three simple forecasts to see what lags add at different horizons.

We compute everything on the **hourly grid** from step 4, so "lag *k*" always means exactly *k* hours earlier, even across gaps.

In [ ]:
# hour-of-day and weekday flags for every column of the grid (Chicago time)
loc = hours.tz_convert(TZ)
grid_hour = np.asarray(loc.hour)
grid_wkday = np.asarray(loc.dayofweek < 5)

def remove_daily_cycle(Y):
    # subtract each sensor's usual value for that hour and weekday/weekend
    Z = Y.copy()
    for h in range(24):
        for w in (True, False):
            cols = (grid_hour == h) & (grid_wkday == w)
            Z[:, cols] -= np.nanmean(Y[:, cols], axis=1, keepdims=True)
    return Z

def pooled_acf(Y, lags):
    # correlation of (t, t-k) pairs pooled over all sensors, skipping missing hours
    out = []
    for k in lags:
        a, b = Y[:, k:].ravel(), Y[:, :-k].ravel()
        ok = ~(np.isnan(a) | np.isnan(b))
        out.append(np.corrcoef(a[ok], b[ok])[0, 1])
    return np.array(out)

NO2 = grid["no2"].to_numpy()
LOGPM = np.log(grid["pm"].to_numpy())
lags = list(range(1, 73)) + [96, 120, 144, 168]
acf = pd.DataFrame({
    "NO₂": pooled_acf(NO2, lags),
    "NO₂, daily cycle removed": pooled_acf(remove_daily_cycle(NO2), lags),
    "log PM2.5": pooled_acf(LOGPM, lags),
}, index=lags)
display(acf.loc[[1, 2, 3, 6, 12, 24, 48, 168]].round(2))

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(acf.index, acf["NO₂"], color=C["no2"], lw=2, label="NO₂")
ax.plot(acf.index, acf["NO₂, daily cycle removed"], color=C["no2"], lw=2, ls="--", label="NO₂, daily cycle removed")
ax.plot(acf.index, acf["log PM2.5"], color=C["pm"], lw=2, label="log PM2.5")
ax.set(ylim=(0, 1), xticks=[1, 12, 24, 48, 72, 96, 120, 144, 168], xlabel="lag (hours)", ylabel="correlation",
       title="Step 10 · Autocorrelation within each sensor")
ax.legend(); save(fig, "s10_acf"); plt.show()

### Three baseline forecasts
We predict the value *h* hours ahead (h = 1, 2, 3, 6, 12, 24) three ways and compare the mean absolute error (MAE) on a test period:
- **Persistence:** "it will be the same as now".
- **Typical hour:** the sensor's average for that hour and weekday/weekend (a time-anchor-only forecast).
- **Lags + typical hour:** a pooled linear regression on the current value, 1 and 2 hours ago, the same hour yesterday, and the typical-hour value.

Training: Sep 2025 – Apr 2026. Test: May 1 – Sep 26 2026. These are baselines to learn from, not a final model.

In [ ]:
test_cols = hours >= pd.Timestamp("2026-05-01", tz=TZ)

def typical_hour(Y):
    # sensor x hour x weekday/weekend mean, learned on the training period only
    clim = np.full_like(Y, np.nan)
    for h in range(24):
        for w in (True, False):
            cols = (grid_hour == h) & (grid_wkday == w)
            clim[:, cols] = np.nanmean(Y[:, cols & ~test_cols], axis=1, keepdims=True)
    return clim

rows = []
for p in ["no2", "pm"]:
    Y = grid[p].to_numpy()
    clim = typical_hour(Y)
    T = Y.shape[1]
    for h in [1, 2, 3, 6, 12, 24]:
        target = Y[:, 24 + h:]
        now, lag1, lag2 = Y[:, 24:T - h], Y[:, 23:T - h - 1], Y[:, 22:T - h - 2]
        yesterday = Y[:, h:T - 24]                          # same clock hour as the target, one day earlier
        X = np.stack([now, lag1, lag2, yesterday, clim[:, 24 + h:]], axis=-1).reshape(-1, 5)
        y = target.ravel()
        is_test = np.broadcast_to(test_cols[24 + h:], target.shape).ravel()
        ok = ~(np.isnan(X).any(axis=1) | np.isnan(y))
        tr, te = ok & ~is_test, ok & is_test
        beta = np.linalg.lstsq(np.c_[np.ones(tr.sum()), X[tr]], y[tr], rcond=None)[0]
        pred = np.c_[np.ones(te.sum()), X[te]] @ beta
        mae = lambda f: np.mean(np.abs(y[te] - f))
        rows.append({"target": p, "horizon (h)": h, "persistence": mae(X[te, 0]),
                     "typical hour": mae(X[te, 4]), "lags + typical hour": mae(pred)})

forecast = pd.DataFrame(rows).set_index(["target", "horizon (h)"]).round(2)
display(forecast)

fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
for ax, p, unit in [(axes[0], "no2", "ppb"), (axes[1], "pm", "µg/m³")]:
    f = forecast.loc[p]
    ax.plot(f.index, f["persistence"], "o-", color=C[p], lw=2, label="persistence")
    ax.plot(f.index, f["typical hour"], "s--", color=C["grey"], lw=1.5, label="typical hour")
    ax.plot(f.index, f["lags + typical hour"], "^-", color="black", lw=1.5, label="lags + typical hour")
    ax.set(xticks=f.index, xlabel="forecast horizon (hours)", ylabel=f"MAE ({unit})",
           title=f"{'NO₂' if p == 'no2' else 'PM2.5'} forecast error (test May–Sep 2026)"); ax.legend()
fig.tight_layout(); save(fig, "s10_forecast_baselines"); plt.show()

In [ ]:
# If PM2.5 AQI is above a threshold now, how likely is it to still be above it h hours later?
Q = grid["pm_aqi"].to_numpy()
res = {}
for thr in [50, 100]:
    base = np.mean(Q[~np.isnan(Q)] > thr)
    col = {"base rate": base}
    for h in [1, 3, 6, 12, 24]:
        now, later = Q[:, :-h], Q[:, h:]
        ok = ~(np.isnan(now) | np.isnan(later))
        col[f"still above after {h} h"] = ((later > thr) & (now > thr) & ok).sum() / ((now > thr) & ok).sum()
        # share of above-threshold hours that were NOT above threshold h hours earlier (= new starts)
        col[f"new starts ({h} h)"] = ((later > thr) & (now <= thr) & ok).sum() / ((later > thr) & ok).sum()
    res[f"AQI > {thr}"] = col
display((pd.DataFrame(res) * 100).round(1))

### Building the lag features (for the modelling notebook)
This is how to create the features safely. `asfreq("h")` inserts the missing hours as empty rows, so `shift(k)` really means *k hours ago*. Rolling means use `shift(1)` first so they only look at the past (no leakage from the hour we're predicting).

In [ ]:
LAGS = [1, 2, 3, 6, 12, 24, 168]
parts = []
for sid, g in A[["sid", "utc", "local", "no2", "pm"]].groupby("sid"):
    g = g.set_index("utc").sort_index().asfreq("h")        # full hourly index; missing hours = NaN
    g["sid"] = sid
    g["local"] = g.index.tz_convert(TZ)
    g["log_pm"] = np.log(g["pm"])
    for k in LAGS:
        g[f"no2_lag{k}"] = g["no2"].shift(k)
        g[f"logpm_lag{k}"] = g["log_pm"].shift(k)
    g["no2_mean_3h"] = g["no2"].shift(1).rolling(3, min_periods=2).mean()
    g["no2_mean_24h"] = g["no2"].shift(1).rolling(24, min_periods=18).mean()
    g["logpm_mean_3h"] = g["log_pm"].shift(1).rolling(3, min_periods=2).mean()
    g["logpm_mean_24h"] = g["log_pm"].shift(1).rolling(24, min_periods=18).mean()
    parts.append(g)
features = pd.concat(parts).reset_index()

# time anchors (Chicago time), with sin/cos so 23:00 and 00:00 end up close together
features["hour"] = features["local"].dt.hour
features["dow"] = features["local"].dt.dayofweek
features["weekend"] = features["dow"] >= 5
features["month"] = features["local"].dt.month
features["hour_sin"] = np.sin(2 * np.pi * features["hour"] / 24)
features["hour_cos"] = np.cos(2 * np.pi * features["hour"] / 24)

print(features.shape)
print("share of rows with no2_lag1 present:", round(features["no2_lag1"].notna().mean(), 4))
features.head()
# to keep it for modelling:  features.to_csv("outputs/eda_bobo/features_hourly.csv", index=False)  (large file)

**What we see**
- **Lag 1 is very strong:** r = 0.91 for NO₂ and 0.95 for log PM2.5. PM2.5 stays correlated for longer (0.57 at 12 h vs 0.35 for NO₂), which fits a regional pollutant.
- NO₂'s ACF drops to about 0.32 around 16 h, then rises to 0.42 at 24 h and 0.33 at 168 h. That bump is the daily and weekly cycle. Once the daily cycle is removed, the bump disappears. So **lag 24 and lag 168 mostly repeat what hour of day and day of week already say**. Use one or the other, or expect them to overlap.
- **Short horizons (1–3 h):** persistence is hard to beat. The 1 h NO₂ error is 1.5 ppb, versus 3.5 ppb for the typical-hour forecast.
- **Longer horizons (6–24 h):** persistence becomes worse than the typical-hour forecast. Combining lags with the typical hour gives the lowest NO₂ error at every horizon (3.0 vs 3.8 ppb at 6 h). Lags and time anchors work best together.
- For PM2.5, the pooled linear baseline doesn't beat persistence at 1–6 h. The test period contains the July episode (values up to 487), and a single linear fit on the raw scale handles those extremes badly. That's another reason to model **log** PM2.5 or treat episodes separately.
- **AQI flags:** an hour above 100 is followed by another above 100 91% of the time after 1 h, 67% after 6 h and 31% after 24 h, against a base rate of 0.9%. But only 8.8% of those hours were at or below 100 an hour earlier, so the hard part is predicting **when an episode starts**. Part of this persistence also comes from the 12-hour NowCast formula. Forecasting the hourly PM2.5 concentration and converting it to AQI afterward gives an honest target.

## Step 11 · Sensor noise: what the collocation sites tell us

**Why:** at each collocation site (and at O'Hare), three identical units sit side by side and breathe the same air, so any difference between them is measurement noise. That noise sets a floor: no model should claim errors smaller than it, and a forecast error close to it is already very good.

In [ ]:
units = df[df["kind"].isin(["collocation", "ohare"])].copy()
units["site"] = units["name"].astype(str).str.replace(r"\s*\d+$", "", regex=True)   # "Mayfair Collocation 2" -> "Mayfair Collocation"

rows = []
for site, g in units.groupby("site"):
    row = {"site": site}
    for p in ["no2", "pm"]:
        w = g.pivot(index="utc", columns="sid", values=p).dropna()         # hours where all 3 units report
        corr = w.corr().to_numpy()
        pairs = np.triu_indices(w.shape[1], 1)
        row[f"{p} r"] = corr[pairs].mean()
        row[f"{p} mean |diff|"] = np.mean([np.mean(np.abs(w.iloc[:, i] - w.iloc[:, j])) for i, j in zip(*pairs)])
        row[f"{p} hours"] = len(w)
    rows.append(row)
display(pd.DataFrame(rows).set_index("site").round(3))

**What we see**
- For PM2.5, co-located units agree very closely (r = 0.93–0.998, mean difference 1.2–1.7 µg/m³).
- For NO₂, they agree less well (r = 0.79–0.93, mean difference 1.6–3.8 ppb). The 1-hour persistence error of 1.5 ppb is already at that noise level, so real gains in NO₂ forecasting will come at horizons of 3 hours or more.
- Schiller Park has both the highest NO₂ and the largest disagreement between units (3.8 ppb). Hourly NO₂ differences of a few ppb between neighbouring sensors shouldn't be treated as real.

## Step 12 · What we carry into feature building

**Cleaning decisions**
- Convert `startofperiod` from **UTC** to `America/Chicago` before building any time feature.
- Use the calibrated `.value` columns. Drop black carbon and the 0.8%-filled ambient weather columns.
- Analysis set: **274 community sensors, Sep 1 2025 onward**. Keep collocation and O'Hare units only for noise checks.
- Build lags on a complete hourly grid per sensor (`asfreq("h")`), never with a plain `shift` on the sorted rows.
- Model PM2.5 on the **log** scale, and flag citywide episode hours (median > 35 µg/m³) so they can be studied separately.

**Features the data supports**
- **Lags:** NO₂ and log PM2.5 at 1, 2 and 3 h (strongest), plus 24 h. Past-only rolling means over 3 h and 24 h.
- **Time anchors:** hour of day (sin/cos), weekend flag or day of week, and month or season. These matter for NO₂ and very little for PM2.5.
- **Sensor level:** each sensor's own long-run average (12.6% of NO₂ variance is between sensors).

**Open questions for the team**
- Target for "unhealthy hour": PM2.5 AQI > 100 is rare (0.9%) and clustered in a few episodes. Do we forecast the concentration and derive the AQI, or classify the rare events directly?
- The July 2026 episode and the Jul 28 – Aug 13 outage both fall in summer 2026. Should the train/test split be by date, by season, or leave out whole episodes?
- Weather (left out here) has to come from a source outside this file, because its ambient columns are almost empty.